# Introduction to Q-Learning
In this module, we continue our discussion of reinforcement learning by introducing Q-learning, a powerful algorithm for learning optimal policies by exploring the world and learning from experience. By the end of this module, you will be able to define and demonstrate mastery of the following key concepts:

* __Markov Decision Process (MDP)__: A mathematical framework for modeling decision-making where outcomes are partly random and partly under the control of a decision-maker, defined by states, actions, rewards, transition probabilities, and a discount factor. Let's review this topic.
* __Value Iteration__: A dynamic programming algorithm that computes the optimal value function for an MDP by iteratively applying the Bellman operator, which updates the value of each state based on the expected rewards and future values of possible actions, converging to the optimal policy. Let's review this topic.
* __Q-learning__: A model-free, off-policy algorithm that learns the optimal action-value function by iteratively updating its value estimates using observed rewards and the best available future estimates, thereby converging to an optimal policy without requiring any model of the environment.

Q-learning is a super interesting algorithm because it allows us to learn the optimal policy without having a model of the environment. However, it does have some limitations, especially in terms of scaling to large (or continuous) state/action spaces, where the number of possible states and actions can be enormous. 

## General Reinforcement Learning Problem
Suppose we have an agent that can be in a state $s \in \mathcal{S}$ and can take an action $a \in \mathcal{A}$. After taking action $a$ in state $s$, the agent receives a reward $r$. But how does the agent learn to choose the best possible action in each state to maximize its cumulative reward over time?

<div>
    <center>
        <img src="figs/Fig-Schematic-RL.svg" width="580"/>
    </center>
</div>

In reinforcement learning, an agent interacts with an environment by observing its current state $s \in \mathcal{S}$, selecting an action $a \in \mathcal{A}$, and receiving a reward that influences its future decisions. We'll explore three distinct approaches to this problem:

* __Bandit algorithms__ operate in stateless environments. On each round, they explore different actions to estimate their rewards and adapt their action-selection strategy based on the outcomes.
* __Multiplicative weights__ also adapt action probabilities based on past performance, but they do so in a principled way that guarantees the algorithm performs nearly as well as the best fixed action in hindsight—even in changing environments.
* __Q-learning__ is a value-based method that estimates the long-term value of each state-action pair, enabling the agent to learn optimal behavior in environments with temporal and sequential dynamics.


These approaches highlight different strategies for learning from interaction, but they all must balance a fundamental challenge in reinforcement learning: the tradeoff between exploring new actions to gather information and exploiting known actions to maximize reward.

___

## Review: Markov Decision Processes (MDPs)
A Markov decision process (MDP) models decision-making in situations where outcomes are partly random and partly under the control of a decision-maker. An MDP consists of the tuple of components $\left(\mathcal{S}, \mathcal{A}, R_{a}\left(s, s^{\prime}\right), T_{a}\left(s,s^{\prime}\right), \gamma\right)$:

### Components of an MDP
* __States__: The state space $\mathcal{S}$ is the set of all possible states $s\in\mathcal{S}$ that a system can exist in. This is the same idea as a Markov model. For example, let's suppose we define our state space as the set of investor moods $\mathcal{S} \equiv \left\{\text{bullish},\text{neutral},\text{bearish}\right\}$.
* __Actions__: The action space $\mathcal{A}$ is the set of all possible actions $a\in\mathcal{A}$ available to the agent, where $\mathcal{A}_{s} \subseteq \mathcal{A}$ is the subset of the action space $\mathcal{A}$ that is accessible from state $s$. In our investor example, the action space could be defined as $\mathcal{A} \equiv \left\{\text{buy},\text{hold},\text{sell}\right\}$.
* __Reward__: A reward $R_{a}\left(s, s^{\prime}\right)$ is received after transitioning from $s\rightarrow{s}^{\prime}$ due to action $a$. For example, this could be the proceeds (or losses) from the sale of shares of asset `XYZ.`
* __Transitions__: The state transition model $T_{a}\left(s,s^{\prime}\right) = P(s_{t+1} = s^{\prime}~|~s_{t}=s,a_{t} = a)$ denotes the probability that action $a$ in state $s$ at time $t$ will result in state $s^{\prime}$ at time $t+1$. This idea is similar to a Markov model, e.g., it has the Markov property but involves both the current state $s$ and action $a$ as conditions to transition to the next state.
* __Discount__: The discount factor $0<\gamma<1$ weighs the future expected utility of choices. The discount factor $\gamma$ is a hyperparameter of various approaches used to model the decision.

Finally, a policy function $\pi:\mathcal{S}\rightarrow\mathcal{A}$ is the mapping from states $s\in\mathcal{S}$ to actions $a\in\mathcal{A}$ used by the agent to solve a decision task. Ultimately, we want to develop an optimal policy function (one that gives us the best possible decisions). 

There are many techniques for developing optimal policy functions. Let's start with possibly the simplest one, the __Monte-Carlo Tree Search (MCTS)__. In this method, we take random actions and see what happens. Then we'll look at __Value Iteration__, a dynamic programming approach that iteratively refines value estimates for states until they converge to the optimal value function.
___

## Review: Value Iteration
The Value Iteration algorithm is a dynamic programming approach to compute the optimal policy for an MDP. Suppose we have an MDP defined by the tuple  $\left(\mathcal{S}, \mathcal{A}, R_{a}\left(s, s^{\prime}\right), T_{a}\left(s,s^{\prime}\right), \gamma\right)$. Value iteration iteratively computes the optimal value (utility) function $U^{\star}$ using a _greedy Bellman backup_ operation at iteration $k$:
$$
\begin{align*}
U_{k+1}(s) & = \underset{a\in\mathcal{A}}{\max}\left(\underbrace{R(s,a)}_{\text{now!}} + \gamma\cdot\underbrace{\sum_{s^{\prime}\in\mathcal{S}}T(s^{\prime}\,\vert\,s,a)\cdot{U}_{k}(s^{\prime})}_{\text{expected future reward}}\right)\quad\forall{s\in\mathcal{S}}\\
\end{align*}
$$
Let's break down the components of this equation:
* The term $R(s,a)$ represents the _immediate reward_ received for taking action $a$ in state $s$.
* The term $\gamma\cdot\sum_{s^{\prime}\in\mathcal{S}}T(s^{\prime}\,\vert\,s,a)\cdot{U}_{k}(s^{\prime})$ represents the expected future utility, discounted by the factor $\gamma$. It accounts for the probability of transitioning to each possible next state $s^{\prime}$ given the current state $s$ and action $a$, and the utility of those states at iteration $k$.
* The max operator $\max(\cdot)$ selects the action $a\in\mathcal{A}$ that _maximizes_ the sum of the immediate reward and the expected future utility. This is a _greedy choice_ that ensures we are always choosing the best action based on current estimates.

### Policy
The value iteration algorithm computes the optimal value (utility) function $U^{\star}$ for each state $s\in\mathcal{S}$. But what we really want is the optimal policy $\pi^{\star}$ that tells us which action to take in each state. We can then use this utility function to derive the optimal state-action-value function $Q^{\star}(s,a)$, which gives the expected utility of taking action $a$ in state $s$ and following the optimal policy thereafter:
$$
\begin{align*}
Q^{\star}(s,a) = \overbrace{R(s,a)}^{\text{immediate return}} + \underbrace{\gamma\cdot\sum_{s^{\prime}\in\mathcal{S}}T(s^{\prime}\,\vert\,s,a)\cdot{U^{\star}}(s^{\prime})}_{\text{discounted future return}}\quad\forall{s\in\mathcal{S},a\in\mathcal{A}}
\end{align*}
$$
by selecting the action $a\in\mathcal{A}$ such that:
$$
\begin{align*}
\pi^{\star}(s) = \underbrace{\underset{a\in\mathcal{A}}{\arg\max}\,Q^{\star}(s,a)}_{\text{best action for every state}}\quad\forall{s\in\mathcal{S}}
\end{align*}
$$

### Algorithm
Let's develop a simple value iteration algorithm for computing the optimal $U^{\star}(s)$ function.

__Initialize__: Given the components of the MDP $\left(\mathcal{S}, \mathcal{A}, R, T, \gamma\right)$, a tolerance parameter $\epsilon$, the maximum number of iterations $T_{\text{max}}$, initialize $U(s)\gets{0}$ for all $s\in\mathcal{S}$, and $\texttt{converged}\gets\texttt{false}$, and the loop counter $t\gets{1}$.

While $\texttt{converged}$ is $\texttt{false}$ __do__:

1. Set $\Delta\gets{0}$.
2. For each $s\in\mathcal{S}$ __do__:
   - Compute backup: $U^{\prime}(s) \gets \underset{a\in\mathcal{A}}{\max}\left(R(s,a) + \gamma\cdot\sum_{s^{\prime}\in\mathcal{S}}T(s^{\prime}\,\vert\,s,a)\cdot{U}(s^{\prime})\right)$
   - Compute maximum change: $\Delta\gets\max\left(\Delta, | U^{\prime}(s) - U(s) |\right)$. We could consider other distance measures here, not just the maximum change.
3. Update: $U(s) \gets U^{\prime}(s)$ for all $s\in\mathcal{S}$, and update the iteration counter $t \gets t + 1$.
4. Check for convergence:
   - If $\Delta\leq\epsilon$ or $t>T_{\text{max}}$, then $\texttt{converged}\gets\texttt{true}$.

We can compute the policy once we have the optimal value $U(s)$ for all the states. However, there is one technical question: how do we know if our iteration algorithm will converge?

### Convergence
Yes! We are guaranteed to converge. Amazing.  Why? Because the Bellman operator is a _contraction operator_, repeated backups will converge to its unique fixed point (proof omitted). 
As the number of iterations increases, the value function is _guaranteed to converge_ to an optimal utility $\lim_{k\rightarrow\infty}U_{k}\rightarrow{U^{\star}}$.The number of iterations (upper bound) needed to converge scales as:
$$
\begin{align*}
T_{\text{max}} &\sim\left(\frac{1}{1-\gamma}\right)\cdot\ln\left(\frac{1}{\epsilon}\right)
\end{align*}
$$
where $\epsilon$ is the desired accuracy of the utility function. Thus, as the discount factor $\gamma$ approaches `1`, the number of iterations required for convergence increases significantly, which is an essential consideration in practice.
* **Long‐term vs. short‐term focus:** When the discount factor $\gamma$ is close to 1, the agent heavily values future rewards, resulting in slower convergence but more globally optimal decisions. Conversely, a smaller $\gamma$ prioritizes immediate rewards, speeding up convergence at the risk of suboptimal long‐run performance.
* **Role of accuracy $\epsilon$:** The tolerance $\varepsilon$ sets how closely the value function must approximate the optimum. A tighter tolerance (smaller $\varepsilon$) yields a more precise policy but requires more iterations to converge.
* **Choosing $\gamma$:** Select $\gamma$ based on your effective planning horizon $H$, using the guideline $\gamma \approx 1 - \tfrac{1}{H}$. For example, if you care about outcomes 100 steps ahead, use $\gamma\approx0.99$; for roughly 10 steps, use $\gamma\approx0.9$. Higher $\gamma$ boosts long‐term performance but slows convergence, while lower $\gamma$ accelerates learning at the expense of foresight.


The interesting thing about this algorithm is that becuase we are using the Bellman operator, and we have a model of the environment, i.e., we know the transition probabilities $T(s^{\prime}\,\vert\,s,a)$ and the rewards $R(s,a)$, we have some strong convergence guarantees. 


__Suppose this wasn't the case__. In many cases, we do not have a model of the environment, or the rewards and transitions are not known in advance. In these cases, we cannot use the Bellman operator to compute the optimal value function $U^{\star}(s)$ directly. Instead, we must learn the value function from experience, i.e., by interacting with the environment and observing the rewards and transitions. This is where Q-learning comes in.
___

## Q-Learning: Theory & Update Rule
Q-learning iteratively estimates the state action-value function $Q(s, a)$ by conducting repeated experiments $t=1,2,\ldots$ in the world $\mathcal{W}$. 
In each experiment, an agent in state $s\in\mathcal{S}$ takes action $a\in\mathcal{A}$, receives a reward $r$, and (potentially) transitions to a new state $s^{\prime}$. After each experiment $t$, the agent updates its estimate of $Q(s, a)$ using the update rule:
$$
\begin{equation*}
Q_{t+1}(s,a)\leftarrow{Q_{t}(s,a)}+\alpha_{t}\cdot\underbrace{\left(r+\gamma\cdot\max_{a^{\prime}\in\mathcal{A}}Q_{t}(s^{\prime},a^{\prime}) - Q_{t}(s,a)\right)}_{\text{new information}}\quad{t = 1,2,3,\ldots}
\end{equation*}
$$
where $0<\alpha_{t} <{1}$ is the learning rate parameter at time $t$, and $0<\gamma<{1}$ is the discount factor. 
We estimate the policy function $\pi:\mathcal{S}\rightarrow\mathcal{A}$ by selecting the action $a$ that maximizes $Q(s,a)$ at each state $s$:
$$
\begin{equation*}
\pi(s) = \arg\max_{a\in\mathcal{A}}Q(s,a)
\end{equation*}
$$

### Algorithm: One-State-at-a-Time Sweep
Let's take a look at a simple Q-learning algorithm for computing the optimal $Q^{\star}(s,a)$ function. This algorithm will process _each state $s\in\mathcal{S}$ in turn_, and for each state, it will repeatedly explore the environment by taking actions and updating the $Q(s,a)$ values until convergence.

__Initialize__: Given the states $\mathcal{S}$ and the actions $\mathcal{A}$, initialize $Q(s,a)$ arbitrarily for all $s\in\mathcal{S}$, and $a\in\mathcal{A}$.
Set the hyperparameters: learning rate $\alpha(t)$, the discount factor $\gamma$, the exploration rate $\epsilon(t)$, and the convergence tolerance $\delta$.
Set $\texttt{converged}\gets\texttt{false}$, the loop counter $t\gets{1}$ and the maximum number of iterations `maxiter`.

For $s\in\mathcal{S}$ __do__:
1. Set: $t\gets{1}$, $\texttt{converged}\gets\texttt{false}$, $\epsilon\gets\epsilon(t)$, and $\alpha\gets\alpha(t)$.
2. While not $\texttt{converged}$ __do__:
    * Explore versus exploit choice. Generate a random number $p\sim\mathcal{U}(0,1)$.
        - If $p\leq\epsilon$, then choose a random (uniform) action $a\gets\mathcal{U}(\mathcal{A})$ (exploration).
        - Otherwise: choose a greedy action $a \gets \text{arg}\max_{a\in\mathcal{A}}{Q_{t}(s,a)}$ (exploitation).
    3. Take action: $(s^{\prime},r)\gets\texttt{world}(s,a)$. We take action $a$ in state $s$ and observe the next state $s^{\prime}$ and the reward $r$.
    4. Update the state-action-value function: 
        - Update: $Q_{t+1}(s,a)\leftarrow{Q_{t}(s,a)}+\alpha\cdot\underbrace{\left(r+\gamma\cdot\overbrace{\max_{a^{\prime}\in\mathcal{A}}Q_{t}(s^{\prime},a^{\prime})}^{\text{one-step lookahead}} - Q_{t}(s,a)\right)}_{\text{new information}}$.
    5. Update the iteration counter $t\gets{t+1}$, the state $s\leftarrow{s^{\prime}}$, the exploration rate $\epsilon\leftarrow\epsilon(t)$, and the learning rate $\alpha\leftarrow\alpha(t)$.
    6. Check for convergence:
     - If the $Q(s,a)$ has a bounded maximum change between iterations: $\max_{s,a}\bigl|Q_{t+1}(s,a)-Q_t(s,a)\bigr|\;\le\;\delta$, then set $\texttt{converged}\gets\texttt{true}$. Otherwise, keep iterating.
     - If $t> \texttt{maxiter}$, we've run out of iterations. Set $\texttt{converged}\gets\texttt{true}$. Move to the next start state $s$.
        > __Optional__: Warn the user that the algorithm has not converged, and suggest increasing the number of iterations or adjusting the learning rate.


### Convergence
Can we say anything about the convergence of this algorithm? Yes! Be prepared to be amazed.

The one-state-at-a-time sweep will converge to the optimal $Q^{\star}$ with probability 1, if a few things are true:
1. **Finite MDP & Discounting:** The state–action space $\mathcal{S}\times\mathcal{A}$ must be _finite_, and the discount factor must satisfy $0 \le \gamma < 1$.
2. **Robbins–Monro Step-Size Schedule:** Your learning‐rate sequence ${\alpha(t)}$ must satisfy
   $$
   \sum_{t=1}^\infty \alpha(t) = \infty,
   \qquad
   \sum_{t=1}^\infty \bigl[\alpha(t)\bigr]^2 < \infty.
   $$
   A common choice is $\alpha(t)=1/(1+t)^\kappa$ with $\tfrac12<\kappa\le1$.
3. **Infinite Exploration:** Every state–action pair $(s,a)$ must be visited (and thus updated) infinitely often (w.p. 1). In practice you need $\epsilon(t)$ decays slowly enough (e.g.\ $\epsilon(t)=\epsilon_0/(1+d\,t)$) so that you continue to explore, and your transition dynamics from each $s$ allow you to return to *that same* $s$ arbitrarily many times (or else you explicitly reset/teleport back).

That last point is a little tricky.  We could to run this algorithm for multiple _episodes_ where we initialize $Q(s,a)$ using the output from the previous episode, i.e., we learn from experience. In each episode, we would explore the environment by taking actions and updating the $Q(s,a)$ values until convergence. This way, we can ensure that we visit every state–action pair (infinitely) often.


### Caveats & Practical Notes
* **Wandering Away:** If your dynamics never revisit the *current* $s$ (so you only update $Q(s,\cdot)$ once per sweep), you may fail the _infinitely‐often_ requirement. You must ensure each sweep actually updates every $(s,a)$ pair infinitely many times—e.g., by _teleporting_ back or bounding sweep length.
* **Exploration vs. Exploitation Trade-off:** If $\epsilon(t)$ decays too fast, you may stop exploring before you’ve visited all pairs enough times; if it decays too slowly, learning will be very noisy. Tuning the decay constant is critical.
* **Rate of Convergence:** We have almost-sure convergence but we can't say *how fast*. In practice, the asynchronous sampling that have been proposed can be slow, especially in large state and action spaces, so many implementations switch to either **replay buffers** or full **value‐iteration sweeps** when a model is known.

Hmmmm. That replay buffer idea sounds interesting. Let's explore that next.

___